In [ ]:
import numpy as np
import matplotlib.pyplot as plt  
import pandas as pd
from IPython.display import display

from tqdm import tqdm

# cross-sectional
loader_cross_sectional_tb = features_loader_cross_sectional_tissue_based.FeaturesLoader

# logitudianl
loader_logitudinal_wbb = features_loader_longitudinal_whole_brain_based.FeaturesLoader
loader_logitudinal_tb = features_loader_longitudinal_tissue_based.FeaturesLoader

#registration
loader_registration_wbb = features_loader_registration_whole_brain_based.FeaturesLoader
loader_registration_tbb = features_loader_registration_tissue_based.FeaturesLoader

# approaches
loader_approaches_wbb = features_loader_approaches_whole_brain_based.FeaturesLoader
loader_approaches_tbb = features_loader_approaches_tissue_based.FeaturesLoader

np.set_printoptions(precision=4)

from scipy import stats

from brainpred import paths, splits
from brainpred.utils import display as vis
from brainpred.utils import intensity, metrics, ml as uml
from brainpred.features.cross_sectional import tissue_based as features_loader_cross_sectional_tissue_based
from brainpred.features.longitudinal import whole_brain_based as features_loader_longitudinal_whole_brain_based
from brainpred.features.longitudinal import tissue_based as features_loader_longitudinal_tissue_based
from brainpred.features.registration import whole_brain_based as features_loader_registration_whole_brain_based
from brainpred.features.registration import tissue_based as features_loader_registration_tissue_based
from brainpred.features.approaches import whole_brain_based as features_loader_approaches_whole_brain_based
from brainpred.features.approaches import tissue_based as features_loader_approaches_tissue_based
from brainpred.features.approaches.common import get_diffusion_model_ids

# folder for the tables written by this notebook (patient-level data: never commit them)
current_path = paths.get_work_folder("analysis")

# Important regions of the brain

- **lateral ventricles** (R:43, L:4)
- **hipocampus** (R:53, L:17)
- **thalamus** (R:49, L:10)
- **cortex** (R:42, L:3)
    - entorhinal cortex (R:2006, L:1006)
    - inferior temporal gyri (R:2009, L:1009)
    - superior parietal (R:2029, L:1029)
    - middle frontal gyrus (R:2027, L:1027)
    - superior frontal gyrus (R:2028, L:1028)
    - precuneus (R:2025, L:1025)
    - cuneus (R:2005, L:1005)

**Other founds:**
- 3rd ventricle (14)

In [ ]:
val_images = get_diffusion_model_ids()
nb_tissues = 3
# IMPORTANT_TISSUES_36_96 = [43, 4, 53, 17, 49, 10, 42, 3, 2006, 1006, 2009, 1009, 2029, 1029, 2027, 1027, 2028, 1028, 2025, 1025, 2005, 1005, 14]
if nb_tissues == 96:
    IMPORTANT_TISSUES = [43, 4, 53, 17, 49, 10, 2006, 1006, 2009, 1009, 2029, 1029, 2027, 1027, 2028, 1028, 2025, 1025, 2005, 1005, 14]
    join_mean = [(43, 4), (53, 17), (49, 10), (2006, 1006), (2009, 1009), (2029, 1029), (2027, 1027), (2028, 1028), (2025, 1025), (2005, 1005)]
elif nb_tissues == 36:
    IMPORTANT_TISSUES = [43, 4, 53, 17, 49, 10, 42, 3, 14]
    join_mean = [(43, 4), (53, 17), (49, 10), (42, 3)]
elif nb_tissues == 3:
    join_mean = [(1, 1), (2, 2), (3, 3)]
    IMPORTANT_TISSUES = [1,2,3]


# Load features

In [ ]:
# ---------------- cross-sectional:
# ------------ 1. tissue-based
data_cross_sectional_tb = loader_cross_sectional_tb(nb_tissues, val_images).get_df()

# display(data_cross_sectional_tb.head())

# ---------------- longitudinal:
data_logitudinal_wbb = features_loader_longitudinal_whole_brain_based.FeaturesLoader().get_df(id_patients=val_images, remove_patients=False)
data_logitudinal_tb = features_loader_longitudinal_tissue_based.FeaturesLoader(nb_tissues, val_images).get_df()

# ---------------- registration:
# data_registration_wbb = features_loader_registration_whole_brain_based.FeaturesLoader().get_df()
# data_registration_tb = features_loader_registration_tissue_based.FeaturesLoader(nb_tissues, val_images).get_df(metrics_name="dice")

# ---------------- approaches:
data_approaches_wbb = {}
data_approaches_tb = {}

# dfield_similar_patches
data_approaches_wbb["diffusion_model"] = features_loader_approaches_whole_brain_based.FeaturesLoader("diffusion_model").get_df()
data_approaches_tb["diffusion_model"] = features_loader_approaches_tissue_based.FeaturesLoader("diffusion_model", nb_tissues, val_images).get_df()


# display(data_approaches_wbb["dfield_similar_patches"].head(10))
# display(data_logitudinal_wbb.head(10))

## t-score

In [ ]:
t_stat_A, p_val_A = stats.ttest_rel(data_approaches_wbb["diffusion_model"]["ssim_hunt4"].values, data_logitudinal_wbb["ssim"].values)


print(f"Metodo A: t = {t_stat_A}, p = {p_val_A}")

# Comparación de resultados
if p_val_A < 0.05:
    print("Método A es significativo.")


## Dice score

In [ ]:
def compute_mean_emisphres(data, join_mean, sufix="dice", desired_sufix=None):
    if desired_sufix is None:
        desired_sufix = sufix
    columns = [f"{t1}_{t2}_{desired_sufix}" for t1, t2 in join_mean]
    _data = pd.DataFrame()
    for i, (t1, t2) in enumerate(join_mean):
        _data[columns[i]] = (data[f"{t1}_{sufix}"] + data[f"{t2}_{sufix}"]) / 2
    return _data


def compute_sum_emisphres(data, join_mean, sufix="vol_hunt3", desired_sufix=None):
    if desired_sufix is None:
        desired_sufix = sufix
    columns = [f"{t1}_{t2}_{desired_sufix}" for t1, t2 in join_mean]
    _data = pd.DataFrame()
    for i, (t1, t2) in enumerate(join_mean):
        _data[columns[i]] =  (data[f"{t1}_{sufix}"] + data[f"{t2}_{sufix}"])
    if join_mean[0][0] == join_mean[0][1]: # just for 3 tissues segmentation
        return _data/2
    return _data


def create_statistic_table(data, initial_data, columns, mult=1):
    res_dict = {}
    res_means_dict = {}
    res_sem_dict = {}
    res_t_test_dict = {}
    res_p_value_dict = {}
    for i, column in enumerate(columns):
        _initial_res = initial_data[column].values * mult
        res_dict[column] = data[column].values * mult

        t_stat_A, p_val_A = stats.ttest_rel(res_dict[column], _initial_res)
        res_t_test_dict[column] = t_stat_A
        res_p_value_dict[column] = p_val_A
        res_means_dict[column] = res_dict[column].mean()
        res_sem_dict[column] = stats.sem(res_dict[column], axis=0)

    # create np table
    res_table = np.zeros((len(columns), 4))
    for i, column in enumerate(columns):
        res_table[i, 0] = res_means_dict[column]
        res_table[i, 1] = res_sem_dict[column]
        res_table[i, 2] = res_t_test_dict[column]
        res_table[i, 3] = res_p_value_dict[column]

    # create pd table
    df_mean = pd.DataFrame(res_table[:,0][:, np.newaxis].T, columns=columns)
    df_sem = pd.DataFrame(res_table[:,1][:, np.newaxis].T, columns=columns)
    df_t_test = pd.DataFrame(res_table[:,2][:, np.newaxis].T, columns=columns)
    df_p_value = pd.DataFrame(res_table[:,3][:, np.newaxis].T, columns=columns)
    return df_mean, df_sem, df_t_test, df_p_value






In [ ]:

def display_dice_table(hunt=4):
    approaches = ["initial", "diffusion_model"]

    mean_table = np.zeros((len(approaches), len(join_mean)))
    sem_table = np.zeros((len(approaches), len(join_mean)))
    t_test_table = np.zeros((len(approaches), len(join_mean)))
    p_value_table = np.zeros((len(approaches), len(join_mean)))

    desired_sufix="dice"
    initial_data = compute_mean_emisphres(data_logitudinal_tb, join_mean, sufix="dice", desired_sufix=desired_sufix)
    # initial_data = compute_sum_emisphres(data_logitudinal_tb, join_mean, sufix="dice", desired_sufix=desired_sufix) / 2
    columns = [f"{t1}_{t2}_{desired_sufix}" for t1, t2 in join_mean]

    for i, approach in enumerate(approaches):
        if approach in [None, "initial"]:
            data = initial_data
        else:
            data = compute_mean_emisphres(data_approaches_tb[approach], join_mean, sufix=f"dice_hunt{hunt}", desired_sufix=desired_sufix)
            # data = compute_sum_emisphres(data_approaches_tb[approach], join_mean, sufix=f"dice_hunt{hunt}", desired_sufix=desired_sufix) / 2
        
        df_mean, df_sem, df_t_test, df_p_value = create_statistic_table(data, initial_data, columns, mult=100)
        
        mean_table[i] = df_mean.values[0]
        sem_table[i] = df_sem.values[0]
        t_test_table[i] = df_t_test.values[0]
        p_value_table[i] = df_p_value.values[0]

    df_mean = pd.DataFrame(mean_table, columns=columns, index=approaches)
    df_sem = pd.DataFrame(sem_table, columns=columns, index=approaches)
    df_t_test = pd.DataFrame(t_test_table, columns=columns, index=approaches)
    df_p_value = pd.DataFrame(p_value_table, columns=columns, index=approaches)
    return df_mean, df_sem, df_t_test, df_p_value

pd.options.display.float_format = '{:.2f}'.format

df_mean, df_sem, df_t_test, df_p_value = display_dice_table()
df_mean_hunt3, df_sem_hunt3, df_t_test_hunt3, df_p_value_hunt3 = display_dice_table(hunt=3)
# df_mean.to_csv(f"{current_path}/dice_mean.csv")
print("means HUNT4")
display(df_mean)
print("means HUNT3")
display(df_mean_hunt3)
# print("std error")
# display(df_sem)
# print("t-test")
# display(df_t_test)
# print("p-value")
# display(df_p_value)

## Volume precentage (increase/decrease)

In [ ]:
def get_vol_percentage(data_base, data, colums):
    _data = pd.DataFrame()
    for column in colums:
        _data[column] = data[column] / data_base[column]
    return _data

def display_volumn_table():
    approaches = ["initial", "diffusion_model"]

    mean_table = np.zeros((len(approaches), len(join_mean)))
    sem_table = np.zeros((len(approaches), len(join_mean)))
    t_test_table = np.zeros((len(approaches), len(join_mean)))
    p_value_table = np.zeros((len(approaches), len(join_mean)))

    desired_sufix="vol"
    data_hunt3 = compute_sum_emisphres(data_cross_sectional_tb, join_mean, sufix="vol_hunt3", desired_sufix=desired_sufix)
    data_hunt4 = compute_sum_emisphres(data_cross_sectional_tb, join_mean, sufix="vol_hunt4", desired_sufix=desired_sufix)
    columns = [f"{t1}_{t2}_{desired_sufix}" for t1, t2 in join_mean]

    initial_data = get_vol_percentage(data_hunt3, data_hunt4, [f"{t1}_{t2}_{desired_sufix}" for t1, t2 in join_mean])
    for i, approach in enumerate(approaches):
        if approach in [None, "initial", "expected"]:
            data = initial_data
        else:
            _data = compute_sum_emisphres(data_approaches_tb[approach], join_mean, sufix="vol", desired_sufix=desired_sufix)
            data = get_vol_percentage(data_hunt3, _data, columns)
            # print(data.min().values, data.max().values)

        df_mean, df_sem, df_t_test, df_p_value = create_statistic_table(data, initial_data, columns, mult=1)
        
        mean_table[i] = df_mean.values[0]
        sem_table[i] = df_sem.values[0]
        t_test_table[i] = df_t_test.values[0]
        p_value_table[i] = df_p_value.values[0]

    df_mean = pd.DataFrame(mean_table, columns=columns, index=approaches)
    df_sem = pd.DataFrame(sem_table, columns=columns, index=approaches)
    df_t_test = pd.DataFrame(t_test_table, columns=columns, index=approaches)
    df_p_value = pd.DataFrame(p_value_table, columns=columns, index=approaches)
    return df_mean, df_sem, df_t_test, df_p_value

pd.options.display.float_format = '{:.5f}'.format

df_mean, df_sem, df_t_test, df_p_value = display_volumn_table()
# df_mean.to_csv(f"{current_path}/dice_mean.csv")
print("means HUNT4")
display(df_mean)

print("std error")
display(df_sem)
print("t-test")
display(df_t_test)
print("p-value")
display(df_p_value)


## vol relative absolute volume difference

In [ ]:

def get_relative_absolute_volume_difference(data_base, data, colums):
    _data = pd.DataFrame()
    for column in colums:
        _data[column] = np.abs(data[column] - data_base[column]) / data_base[column]
    return _data

def display_volumn_table():
    approaches = ["initial", "diffusion_model"]

    mean_table = np.zeros((len(approaches), len(join_mean)))
    sem_table = np.zeros((len(approaches), len(join_mean)))
    t_test_table = np.zeros((len(approaches), len(join_mean)))
    p_value_table = np.zeros((len(approaches), len(join_mean)))

    desired_sufix="vol"
    data_hunt3 = compute_sum_emisphres(data_cross_sectional_tb, join_mean, sufix="vol_hunt3", desired_sufix=desired_sufix)
    data_hunt4 = compute_sum_emisphres(data_cross_sectional_tb, join_mean, sufix="vol_hunt4", desired_sufix=desired_sufix)
    columns = [f"{t1}_{t2}_{desired_sufix}" for t1, t2 in join_mean]

    initial_data = get_relative_absolute_volume_difference(data_hunt4, data_hunt3, [f"{t1}_{t2}_{desired_sufix}" for t1, t2 in join_mean])
    for i, approach in enumerate(approaches):
        if approach in [None, "initial", "expected"]:
            data = initial_data
        else:
            _data = compute_sum_emisphres(data_approaches_tb[approach], join_mean, sufix="vol", desired_sufix=desired_sufix)
            data = get_relative_absolute_volume_difference(data_hunt4, _data, columns)

        df_mean, df_sem, df_t_test, df_p_value = create_statistic_table(data, initial_data, columns, mult=100)
        
        mean_table[i] = df_mean.values[0]
        sem_table[i] = df_sem.values[0]
        t_test_table[i] = df_t_test.values[0]
        p_value_table[i] = df_p_value.values[0]

    df_mean = pd.DataFrame(mean_table, columns=columns, index=approaches)
    df_sem = pd.DataFrame(sem_table, columns=columns, index=approaches)
    df_t_test = pd.DataFrame(t_test_table, columns=columns, index=approaches)
    df_p_value = pd.DataFrame(p_value_table, columns=columns, index=approaches)
    return df_mean, df_sem, df_t_test, df_p_value

pd.options.display.float_format = '{:.5f}'.format

df_mean, df_sem, df_t_test, df_p_value = display_volumn_table()
# df_mean.to_csv(f"{current_path}/dice_mean.csv")
print("means HUNT4")
display(df_mean)

print("std error")
display(df_sem)
print("t-test")
display(df_t_test)
print("p-value")
display(df_p_value)


## brain parechimalfraction


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

def dislay_bpf_table(df_bpf, tytle="BPF", melted=False):
    if not melted:
        df_melted = df_bpf.melt(var_name='Approach', value_name='BPF')
    else:
        df_melted = df_bpf
    # Crear el boxplot con jitter
    plt.figure(figsize=(20, 6))
    sns.boxplot(x='Approach', y='BPF', data=df_melted, palette="pastel")
    sns.stripplot(x='Approach', y='BPF', data=df_melted, color='black', alpha=0.3, jitter=True)
    plt.title(tytle)
    plt.show()

def read_bpf_table():
    columns = ["hunt3", "hunt4", "diffusion_model"]
    desired_columns = ["bpf_hunt3", "bpf_hunt4", "diffusion_model"]
    bpf_table = np.zeros((len(val_images), len(columns)))
    for i, column in enumerate(columns):
        if column in ["hunt3", "hunt4"]:
            data = data_logitudinal_wbb[f"bpf_{column}"].values
        else:
            data = data_approaches_wbb[column]["bpf"].values
        bpf_table[:, i] = data
    df_bpf = pd.DataFrame(bpf_table, columns=desired_columns)
    return df_bpf

def display_all_bpf():
    df_bpf = read_bpf_table()
    dislay_bpf_table(df_bpf, tytle="ALL") 

def display_category_bpf():
    df_bpf = read_bpf_table()

    approaches_columns = df_bpf.columns
    # Calcular los percentiles 33 y 66 para "bpf_hunt3"
    percentil_33 = np.percentile(df_bpf['bpf_hunt3'], 33)
    percentil_66 = np.percentile(df_bpf['bpf_hunt3'], 66)

    # Asignar categorías basado en "bpf_hunt3"
    def categorize_sample(sample):
        if sample <= percentil_33:
            return 'low'
        elif sample <= percentil_66:
            return 'mean'
        else:
            return 'high'
        
    df_bpf['category'] = df_bpf['bpf_hunt3'].apply(categorize_sample)
    combined_data = []

    for index, row in df_bpf.iterrows():
        for approach_name in approaches_columns:
              combined_data.append((row['category'] + '_' + approach_name, row[approach_name]))

    df_combined = pd.DataFrame(combined_data, columns=['Approach', 'BPF'])
    dislay_bpf_table(df_combined, tytle="ALL", melted=True) 


display_all_bpf()
display_category_bpf()

In [ ]:
columns_important_tissues_org = [f"{i}_dice" for i in IMPORTANT_TISSUES]
columns_important_tissues_gen = [f"{i}_dice_hunt4" for i in IMPORTANT_TISSUES]
columns_important_tissues_div = [f"{i}_div" for i in IMPORTANT_TISSUES]

df_comparation = data_logitudinal_tb[["patient_id"]].copy()
df_comparation[columns_important_tissues_org] = data_logitudinal_tb[columns_important_tissues_org]
df_comparation[columns_important_tissues_gen] = data_approaches_tb["dfield_similar_patches"][columns_important_tissues_gen]

df_comparation[columns_important_tissues_div] = data_logitudinal_tb[columns_important_tissues_org].values / data_approaches_tb["dfield_similar_patches"][columns_important_tissues_gen].values
# mask = df_comparation[columns_important_tissues_div] >= 1
# print(mask.head(30))
# count_ge_1 = mask.sum()
# print(count_ge_1)

# display(df_comparation.head(2))
# display(df_comparation[["patient_id"] + columns_div].head(20))
print("BPF MEAN:\n", df_comparation[columns_important_tissues_gen].mean()[1:])
print("\nBPF MAX:\n", df_comparation.max()[1:])

# display(data_logitudinal_tb)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Crear datos de ejemplo
np.random.seed(10)
data = pd.DataFrame({
    'category': np.repeat(['A', 'B', 'C', 'D'], 10),
    'value': np.concatenate([np.random.normal(10, 5, 10),
                             np.random.normal(15, 5, 10),
                             np.random.normal(25, 5, 10),
                             np.random.normal(5, 2, 10)])
})
display(data.head(100))
# Crear el boxplot con jitter
plt.figure(figsize=(10, 6))
sns.boxplot(x='category', y='value', data=data, palette="pastel")
sns.stripplot(x='category', y='value', data=data, color='black', alpha=0.3, jitter=True)
plt.title('A boxplot with jitter')
plt.show()
